## Raw dosya kontrolü (P2-A2-S2b)

Amaç: Volume'a yüklenen 8 Berka dosyasının byte boyutlarını lokal boyutlarla karşılaştırmak.

Çıktı: Her dosya için Volume'daki boyut, beklenen boyut ve durum (OK / FARK / BEKLENMEYEN DOSYA). En altta toplam dosya sayısı.

In [0]:
raw_path = "/Volumes/workspace/bronze/raw/"

# Lokalde `ls -la` ile ölçülen byte boyutları (P2-A2-S1b)
expected_sizes = {
    "account.csv": 155356,
    "card.csv": 31588,
    "client.csv": 94820,
    "disp.csv": 129716,
    "district.csv": 6574,
    "loan.csv": 27037,
    "order.csv": 273800,
    "trans.csv": 69406578,
}

file_list = dbutils.fs.ls(raw_path)

for file_info in file_list:
    file_name = file_info.name
    actual_size = file_info.size

    if file_name not in expected_sizes:
        status = "BEKLENMEYEN DOSYA"
        expected_size = None
    else:
        expected_size = expected_sizes[file_name]
        if actual_size == expected_size:
            status = "OK"
        else:
            status = "FARK"

    print(f"{file_name:15} volume={actual_size:>10}  expected={expected_size}  {status}")

print(f"\nToplam dosya: {len(file_list)} (beklenen: {len(expected_sizes)})")

## P2-A2-S3a: Spark ile okuma ve kolon isimleri

Amaç: 8 dosyayı `header=True` ve `sep=";"` ile okuyup kolon isimlerini kontrol etmek. `inferSchema` kullanılmıyor, yani tüm kolonlar string geliyor ve ham değerler olduğu gibi görünüyor.

Çıktı: Her tablo için kolon sayısı ve kolon isimleri. Kolon isimleri tırnaksız ve düzgün geliyorsa header, delimiter ve quote doğru demektir. `district` tablosunda `A1`…`A16` görünmeli.

In [0]:
raw_path = "/Volumes/workspace/bronze/raw/"
table_names = ["account", "card", "client", "disp", "district", "loan", "order", "trans"]

# Tablo adı -> DataFrame; sonraki hücrelerde tekrar okumamak için
raw_dfs = {}

for table_name in table_names:
    file_path = raw_path + table_name + ".csv"
    df_raw = spark.read.option("header", True).option("sep", ";").csv(file_path)
    raw_dfs[table_name] = df_raw
    print(table_name, "->", len(df_raw.columns), "columns:", df_raw.columns)

## P2-A2-S3b: Satır sayıları

Amaç: Spark'ın okuduğu satır sayılarını, Git Bash'te `wc -l` ile bulunan beklenen değerlerle (header hariç) karşılaştırmak.

Çıktı: Her tablo için Spark sayısı, beklenen sayı ve durum. Hepsi `OK` olmalı ve toplam 1.079.680 çıkmalı. `DIFF` çıkarsa bir satır ya bölünmüş ya da birleşmiş okunuyor demektir (quote ya da line ending sorunu).

In [0]:
expected_counts = {
    "account": 4500,
    "card": 892,
    "client": 5369,
    "disp": 5369,
    "district": 77,
    "loan": 682,
    "order": 6471,
    "trans": 1056320,
}

total_spark = 0

for table_name in table_names:
    spark_count = raw_dfs[table_name].count()
    expected_count = expected_counts[table_name]
    total_spark = total_spark + spark_count

    if spark_count == expected_count:
        status = "OK"
    else:
        status = "DIFF"

    print(table_name, "| spark:", spark_count, "| expected:", expected_count, "|", status)

print("Total:", total_spark, "| expected: 1079680")

## P2-A2-S3c: trans tablosunda boş değerler

Amaç: Ham dosyadaki `""` ve `;;` değerlerinin Spark'ta NULL olarak mı, boş string olarak mı okunduğunu görmek. Sadece boşluk içeren değerler de ayrıca sayılıyor.

Çıktı: Her kolon için null, empty ve whitespace sayıları. Sayılar bronze'da ham hali korumak, silver'da ise boş değerleri tek tipe (NULL) çevirmek için kullanılacak.

In [0]:
from pyspark.sql import functions as F

df_trans = raw_dfs["trans"]

for column_name in df_trans.columns:
    col = F.col(column_name)
    null_count = df_trans.filter(col.isNull()).count()
    empty_count = df_trans.filter(col == "").count()
    whitespace_count = df_trans.filter((F.trim(col) == "") & (col != "")).count()
    print(column_name, "| null:", null_count, "| empty:", empty_count, "| whitespace:", whitespace_count)


## P2-A2-S3d: Tablolar arası key eşleşmesi (orphan kontrolü)

Amaç: Her child tablodaki foreign key'in parent tabloda karşılığı var mı, kontrol etmek. `left_anti` join kullanılıyor: soldaki (child) tablodan sadece sağda (parent) eşleşmesi olmayan satırlar döner.

Çıktı: Her ilişki için orphan sayısı. Hepsi 0 olmalı. 0'dan büyükse o ilişki silver'da ayrıca ele alınacak.

In [0]:
# (child tablo, child kolon, parent tablo, parent kolon)
relations = [
    ("client", "district_id", "district", "A1"),
    ("account", "district_id", "district", "A1"),
    ("disp", "client_id", "client", "client_id"),
    ("disp", "account_id", "account", "account_id"),
    ("card", "disp_id", "disp", "disp_id"),
    ("loan", "account_id", "account", "account_id"),
    ("order", "account_id", "account", "account_id"),
    ("trans", "account_id", "account", "account_id"),
]

for relation in relations:
    child_table = relation[0]
    child_col = relation[1]
    parent_table = relation[2]
    parent_col = relation[3]

    df_child = raw_dfs[child_table]
    df_parent = raw_dfs[parent_table]

    join_condition = df_child[child_col] == df_parent[parent_col]
    df_orphans = df_child.join(df_parent, join_condition, "left_anti")
    orphan_count = df_orphans.count()

    print(child_table + "." + child_col, "->", parent_table + "." + parent_col, "| orphans:", orphan_count)